In [26]:
import pandas as pd

#read in the local flat file downloaded from Kaggle
games_details = pd.read_csv('games_details.csv', low_memory=False)
games_details.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS
0,22200477,1610612759,SAS,San Antonio,1629641,Romeo Langford,Romeo,F,NaN,18:06,...,1.0,1.0,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0
1,22200477,1610612759,SAS,San Antonio,1631110,Jeremy Sochan,Jeremy,F,NaN,31:01,...,6.0,3.0,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0
2,22200477,1610612759,SAS,San Antonio,1627751,Jakob Poeltl,Jakob,C,NaN,21:42,...,1.0,3.0,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0
3,22200477,1610612759,SAS,San Antonio,1630170,Devin Vassell,Devin,G,NaN,30:20,...,0.0,9.0,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0
4,22200477,1610612759,SAS,San Antonio,1630200,Tre Jones,Tre,G,NaN,27:44,...,0.0,2.0,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0


In [27]:
#get list of column headers
print(games_details.columns)

Index(['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_CITY', 'PLAYER_ID',
       'PLAYER_NAME', 'NICKNAME', 'START_POSITION', 'COMMENT', 'MIN', 'FGM',
       'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT',
       'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS',
       'PLUS_MINUS'],
      dtype='object')


In [28]:
#get a snippet view of the full rows
print(games_details.iloc[51:60])

     GAME_ID     TEAM_ID TEAM_ABBREVIATION  TEAM_CITY  PLAYER_ID  \
51  22200478  1610612762               UTA       Utah    1630695   
52  22200466  1610612749               MIL  Milwaukee    1630699   
53  22200466  1610612749               MIL  Milwaukee     203507   
54  22200466  1610612749               MIL  Milwaukee     201572   
55  22200466  1610612749               MIL  Milwaukee    1628960   
56  22200466  1610612749               MIL  Milwaukee     201950   
57  22200466  1610612749               MIL  Milwaukee    1626171   
58  22200466  1610612749               MIL  Milwaukee    1626192   
59  22200466  1610612749               MIL  Milwaukee     204060   

              PLAYER_NAME NICKNAME START_POSITION                 COMMENT  \
51           Micah Potter    Micah            NaN  DNP - Coach's Decision   
52       MarJon Beauchamp   MarJon              F                     NaN   
53  Giannis Antetokounmpo  Giannis              F                     NaN   
54         

# Transformation 1: Standardize Player Names

## Create a new column that shows player name cleaned up to all lower letters and remove any punctuation

In [29]:
games_details['Player_Name_Clean'] = games_details['PLAYER_NAME'].str.lower().str.strip().replace(',', '').str.replace('.', '')
games_details.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean
0,22200477,1610612759,SAS,San Antonio,1629641,Romeo Langford,Romeo,F,NaN,18:06,...,1.0,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0,romeo langford
1,22200477,1610612759,SAS,San Antonio,1631110,Jeremy Sochan,Jeremy,F,NaN,31:01,...,3.0,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0,jeremy sochan
2,22200477,1610612759,SAS,San Antonio,1627751,Jakob Poeltl,Jakob,C,NaN,21:42,...,3.0,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0,jakob poeltl
3,22200477,1610612759,SAS,San Antonio,1630170,Devin Vassell,Devin,G,NaN,30:20,...,9.0,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0,devin vassell
4,22200477,1610612759,SAS,San Antonio,1630200,Tre Jones,Tre,G,NaN,27:44,...,2.0,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0,tre jones


# Transformation 2: Handle missing or incomplete stats

## Standardize game minutes column to remove NaNs and replace with "00:00"

In [30]:
games_details['MIN'] = games_details['MIN'].fillna("00:00")

print(f"How many remaining NaN in MIN column: {games_details['MIN'].isna().sum()}\n")
print(f"Preview of MIN column with zeroes:\n{games_details['MIN'].iloc[51:60]}")

How many remaining NaN in MIN column: 0

Preview of MIN column with zeroes:
51    00:00
52    20:12
53    39:36
54    27:42
55    34:01
56    37:02
57    19:00
58    19:54
59    16:28
Name: MIN, dtype: object


# Transformation 3: Create a new column for whole team name from abbreviation

In [31]:
#Step 1: get the abbreviated team names used in the dataset
unique_values = games_details['TEAM_ABBREVIATION'].unique()
unique_values

array(['SAS', 'NOP', 'WAS', 'UTA', 'MIL', 'CLE', 'DET', 'PHI', 'CHI',
       'ATL', 'IND', 'BOS', 'GSW', 'BKN', 'TOR', 'NYK', 'ORL', 'HOU',
       'DAL', 'MIN', 'POR', 'OKC', 'LAL', 'SAC', 'CHA', 'LAC', 'MIA',
       'PHX', 'MEM', 'DEN', 'NOH', 'NJN', 'SEA', 'NOK'], dtype=object)

In [32]:
#Step 2: created a dictionary for the full team names
team_full_name = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

#Step 3: create new column with team full names from dictionary
games_details['Team_Name_Whole'] = games_details['TEAM_ABBREVIATION'].map(team_full_name)
games_details.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole
0,22200477,1610612759,SAS,San Antonio,1629641,Romeo Langford,Romeo,F,NaN,18:06,...,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0,romeo langford,San Antonio Spurs
1,22200477,1610612759,SAS,San Antonio,1631110,Jeremy Sochan,Jeremy,F,NaN,31:01,...,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0,jeremy sochan,San Antonio Spurs
2,22200477,1610612759,SAS,San Antonio,1627751,Jakob Poeltl,Jakob,C,NaN,21:42,...,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0,jakob poeltl,San Antonio Spurs
3,22200477,1610612759,SAS,San Antonio,1630170,Devin Vassell,Devin,G,NaN,30:20,...,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0,devin vassell,San Antonio Spurs
4,22200477,1610612759,SAS,San Antonio,1630200,Tre Jones,Tre,G,NaN,27:44,...,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0,tre jones,San Antonio Spurs


# Transformation 4: Create a new dataframe, removing un-needed columns

In [33]:
games_details2 = games_details.drop(columns=['TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_CITY', 'PLAYER_ID', 'NICKNAME', 'START_POSITION', 'COMMENT'])
games_details2.head()

,GAME_ID,PLAYER_NAME,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole
0,22200477,Romeo Langford,18:06,1.0,1.0,1.000,0.0,0.0,0.000,0.0,...,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0,romeo langford,San Antonio Spurs
1,22200477,Jeremy Sochan,31:01,7.0,14.0,0.500,2.0,4.0,0.500,7.0,...,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0,jeremy sochan,San Antonio Spurs
2,22200477,Jakob Poeltl,21:42,6.0,9.0,0.667,0.0,0.0,0.000,1.0,...,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0,jakob poeltl,San Antonio Spurs
3,22200477,Devin Vassell,30:20,4.0,13.0,0.308,1.0,6.0,0.167,1.0,...,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0,devin vassell,San Antonio Spurs
4,22200477,Tre Jones,27:44,7.0,12.0,0.583,1.0,3.0,0.333,4.0,...,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0,tre jones,San Antonio Spurs


In [34]:
#new dataframe columns
print(games_details2.columns)

Index(['GAME_ID', 'PLAYER_NAME', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A',
       'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL',
       'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'Player_Name_Clean',
       'Team_Name_Whole'],
      dtype='object')


# Transformation 5: Find Duplicates

In [35]:
#Step 1: find and display duplicate rows
duplicates = games_details2[games_details2.duplicated(keep=False)]
duplicates

,GAME_ID,PLAYER_NAME,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole
65905,22000069,Grayson Allen,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,grayson allen,Memphis Grizzlies
65906,22000069,Jontay Porter,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,jontay porter,Memphis Grizzlies
65907,22000069,Xavier Tillman,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,xavier tillman,Memphis Grizzlies
65919,22000069,Vernon Carey Jr.,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vernon carey jr,Charlotte Hornets
65920,22000069,Malik Monk,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malik monk,Charlotte Hornets
65932,22000070,Tacko Fall,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tacko fall,Boston Celtics
65933,22000070,Aaron Nesmith,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,aaron nesmith,Boston Celtics
65934,22000070,Tremont Waters,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tremont waters,Boston Celtics
65945,22000070,Wayne Ellington,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wayne ellington,Detroit Pistons
65946,22000070,Saben Lee,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,saben lee,Detroit Pistons


In [36]:
num_duplicates_before = games_details2.duplicated().sum()
print(f"Number of duplicate rows before transformation: {num_duplicates_before}")

Number of duplicate rows before transformation: 22


In [37]:
#Step 2: drop dulicate rows
games_details3 = games_details2.drop_duplicates()

num_duplicates_after = games_details3.duplicated().sum()
print(f"Number of duplicate rows after transformation: {num_duplicates_after}")

Number of duplicate rows after transformation: 0


# Transformation 6: Remove rows with 00:00 playing time

## No playing time means no stats, so removing those rows

In [40]:
#Step 1: how many rows have 00:00 in MIN column?
games_details3[games_details3['MIN'] == '00:00'].shape[0]

109669

In [39]:
#preview the rows with 00:00 in MIN column
games_details3[games_details3['MIN'] == '00:00']

,GAME_ID,PLAYER_NAME,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole
13,22200477,Gorgui Dieng,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gorgui dieng,San Antonio Spurs
51,22200478,Micah Potter,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,micah potter,Utah Jazz
62,22200466,Thanasis Antetokounmpo,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,thanasis antetokounmpo,Milwaukee Bucks
63,22200466,George Hill,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,george hill,Milwaukee Bucks
64,22200466,Serge Ibaka,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,serge ibaka,Milwaukee Bucks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665946,21200003,Robert Sacre,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,robert sacre,Los Angeles Lakers
665958,21200001,Cartier Martin,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cartier martin,Washington Wizards
665969,21200001,Omri Casspi,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,omri casspi,Cleveland Cavaliers
665970,21200001,Jon Leuer,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,jon leuer,Cleveland Cavaliers


In [41]:
#Step 2: remove rows with MIN = 00:00
games_details4 = games_details3[games_details3['MIN'] != '00:00']

#Step 3: any rows left with 00:00 in MIN col?
games_details4[games_details4['MIN'] == '00:00']

,GAME_ID,PLAYER_NAME,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole


In [42]:
#here is the current dataset after all transformations
games_details4

,GAME_ID,PLAYER_NAME,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,Player_Name_Clean,Team_Name_Whole
0,22200477,Romeo Langford,18:06,1.0,1.0,1.000,0.0,0.0,0.000,0.0,...,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0,romeo langford,San Antonio Spurs
1,22200477,Jeremy Sochan,31:01,7.0,14.0,0.500,2.0,4.0,0.500,7.0,...,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0,jeremy sochan,San Antonio Spurs
2,22200477,Jakob Poeltl,21:42,6.0,9.0,0.667,0.0,0.0,0.000,1.0,...,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0,jakob poeltl,San Antonio Spurs
3,22200477,Devin Vassell,30:20,4.0,13.0,0.308,1.0,6.0,0.167,1.0,...,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0,devin vassell,San Antonio Spurs
4,22200477,Tre Jones,27:44,7.0,12.0,0.583,1.0,3.0,0.333,4.0,...,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0,tre jones,San Antonio Spurs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668623,11200005,Jordan Hamilton,19,4.0,9.0,0.444,3.0,6.0,0.500,6.0,...,2.0,0.0,2.0,0.0,1.0,3.0,17.0,NaN,jordan hamilton,Denver Nuggets
668624,11200005,Kenneth Faried,23,7.0,11.0,0.636,0.0,0.0,0.000,4.0,...,1.0,1.0,1.0,0.0,3.0,3.0,18.0,NaN,kenneth faried,Denver Nuggets
668625,11200005,Kosta Koufos,15,3.0,7.0,0.429,0.0,0.0,0.000,0.0,...,8.0,0.0,1.0,0.0,0.0,3.0,6.0,NaN,kosta koufos,Denver Nuggets
668626,11200005,Timofey Mozgov,19,1.0,1.0,1.000,0.0,0.0,0.000,0.0,...,3.0,1.0,0.0,0.0,4.0,2.0,2.0,NaN,timofey mozgov,Denver Nuggets


# Paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed

The transformations to the flat file included: standardizing player names, standardizing the Game Minutes column, created a column for the proper whole team name, removed any unneccesary columns, removed any duplicate rows, and removed rows with 00:00 Game Minutes played.  At this time, there are no legal or regulatory guidelines for this specific project.  A noted risk is the transformations for removing "unneccesary" columns, duplicate rows, and rows with 00:00 minutes played could unintentinally distort a player's performance record.  One key assumption so far is that games with 00:00 minutes played will have no game stats.  The Kaggle dataset author states that the data is sourced from NBA games on the NBA website (nba.com/stats).  In addition, the author references their Github page for additional details on data sourcing.  There are no concerns with ethics regarding the collection of data, as it was originally compiled by the NBA for its league.  Mitigation of ethical concerns will include setting clear definitions and thresholds (example - determining what stats-based performance means), ensure that any transformed or replaced data is clear and documented, and identifying non-data factors contributing to this work (example - intangible factors include popularity, skill, injury).